# 07 Lab — Advanced Structures: Ratios, Backspreads, BWB, Jade Lizard

This lab builds each asymmetric structure on the DEMO chain (spot 100, 45 DTE) and locates where the
risk was moved. You will:

1. Build a 1x2 call ratio and see its **unbounded** tail via `analyzer.max_loss`.
2. Build the mirror-image call backspread and find its **defined** valley.
3. Compare a broken-wing butterfly against a regular butterfly with `viz.plot_compare`.
4. Check the jade lizard's **credit vs call-spread width** upside-risk rule.

Runs offline, top-to-bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import strategies, analyzer, payoff, viz, data

SPOT, EXP = 100.0, 45/365
chain = data.load_sample_chain("DEMO")

## 1. Call ratio spread (1x2): a naked tail

Long one 100 call (3.91), short two 105 calls (1.85 each). Near even money — but the second short
call is naked to the upside.

In [ ]:
ratio = strategies.call_ratio_spread(
    long_call=(100.0, 3.91), short_call=(105.0, 1.85),
    expiry=EXP, ratio=(1, 2),
)
print(ratio.describe())
print("net premium $:", ratio.net_premium())
print("max_profit $:", analyzer.max_profit(ratio))
print("max_loss $:", analyzer.max_loss(ratio))

`max_loss` is `-inf`: the extra short call makes upside losses unbounded. Max profit sits
at the short strike (105). Draw the expiry payoff to see the wide no-loss zone and the tail.

In [ ]:
spots = np.linspace(85, 125, 161)
ax = viz.plot_payoff(ratio, spots)
ax.set_title("Call ratio 1x2: profit tent, then unbounded tail up")

## 2. Call backspread: the mirror image

Short one 100 call (3.91), long two 105 calls (1.85). Now you are **net long** options: a defined
valley of loss, then convex profit on a big up-move.

In [ ]:
back = strategies.call_backspread(
    short_call=(100.0, 3.91), long_call=(105.0, 1.85),
    expiry=EXP, ratio=(1, 2),
)
print("net premium $ (- = credit):", back.net_premium())
print("max_loss $ (defined):", analyzer.max_loss(back))

In [ ]:
pnl = payoff.pnl_curve(back, spots)
valley = spots[np.argmin(pnl)]
print("valley bottom near spot:", round(valley, 1), "loss $:", round(pnl.min(), 2))
ax = viz.plot_payoff(back, spots); ax.set_title("Call backspread: defined valley, convex upside")

The valley bottoms at the long strike (105); above it the two longs overpower the one
short and profit runs convex. Loss is **defined** because the naked leg is gone.

## 3. Broken-wing butterfly vs regular butterfly

Regular put fly +1/-2/+1 with equal 2.5-wide wings (95/97.5/100). BWB with a wider lower wing
(92.5/97.5/100) to cheapen the far side and skew the risk.

In [ ]:
reg_fly = strategies.long_put_butterfly(
    low=(95.0, 1.58), mid=(97.5, 2.37), high=(100.0, 3.42), expiry=EXP)
bwb = strategies.broken_wing_butterfly(
    "put", low=(92.5, 1.01), mid=(97.5, 2.37), high=(100.0, 3.42), expiry=EXP)
print("regular fly net premium $:", reg_fly.net_premium())
print("BWB net premium $:", bwb.net_premium())

In [ ]:
ax = viz.plot_compare([reg_fly, bwb])
ax.set_title("Regular fly (symmetric) vs broken-wing (skewed, riskless upside)")

The BWB flattens on the upside (riskless side — all puts expire worthless, keep the credit)
and carries its defined loss over the wider lower wing. Confirm with the analyzer.

In [ ]:
for name, p in [("regular", reg_fly), ("BWB", bwb)]:
    print(name, "max_profit", round(analyzer.max_profit(p), 1),
          "max_loss", round(analyzer.max_loss(p), 1))

## 4. Jade lizard: the upside-risk check

Short 95 put (1.58), short 105 call (1.85), long 110 call (0.73). The rule: **no upside risk only if
total credit >= call-spread width**. Verify it explicitly.

In [ ]:
jade = strategies.jade_lizard(
    short_put=(95.0, 1.58), short_call=(105.0, 1.85), long_call=(110.0, 0.73), expiry=EXP)
credit = -jade.net_premium() / 100          # per share, credit positive
width = 110.0 - 105.0
print("credit/share:", round(credit, 2), "call-spread width:", width)
print("upside risk eliminated?", credit >= width)

Here credit (2.70) < width (5.0), so this lizard **still has upside risk**. The analyzer's
`max_profit` on the upper plateau equals the credit; a wide up-move loses (width - credit). Inspect
the payoff and summary.

In [ ]:
print(analyzer.summarize(jade, SPOT, vol=0.26))
ax = viz.plot_payoff(jade, np.linspace(80, 120, 161))
ax.set_title("Jade lizard (credit < width -> upside risk remains)")

## Experiments

1. In the ratio, move the short strike to 110 (0.73 each). Does the tail get cheaper or more
   dangerous, and what happens to the no-loss zone width?
2. Re-price the backspread with a lower flat `vol` in `analyzer.summarize`. Backspreads are long
   vega — how does cheaper vol change its appeal at entry?
3. Break the BWB's *upper* wing instead of the lower (e.g. low=95, mid=97.5, high=102.5 in calls).
   Which side is riskless now?
4. Fix the jade lizard's upside risk: narrow the call spread to width 2.5 (sell 105 / buy 107.5) and
   re-check `credit >= width`. What credit do you need?
5. Overlay the ratio and the backspread with `viz.plot_compare` — they are mirror images built from
   the same three strikes.